# BirdCLEF 2026 — perch-label-head + quantile_mix + slide_max  ★ BEST

- **PP**: quantile_mix(α=0.5) → slide_max(w=0.5, win=9) (BEST combo)
- **Local**: 0.4900 → **0.5085** (+0.0185)
- **Weights**: `birdclef2026-perch-label-head`


In [ ]:
import subprocess, sys

# Install TF 2.20 if needed (Kaggle ships 2.19 which breaks Perch v2)
try:
    import tensorflow as tf
    assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 20)
except (ImportError, AssertionError):
    import os
    tb = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl'
    tf_ = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
    for w in [tb, tf_]:
        if os.path.isfile(w):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', w], check=False)

import tensorflow as tf
print(f'TensorFlow {tf.__version__}')
assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 20)
tf.experimental.numpy.experimental_enable_numpy_behavior()

In [ ]:
import glob, os, re, time
from concurrent.futures import ThreadPoolExecutor
import h5py
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

START          = time.time()
TERMINATE_TIME = START + 5300   # 88 min — leave 2 min buffer for submission write

## Config

In [ ]:
DATA_DIR      = '/kaggle/input/birdclef-2026'
PERCH_DIR     = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
WEIGHTS_PATH  = '/kaggle/input/birdclef2026-perch-label-head/best_head.weights.h5'

ALPHA         = 0.4   # Perch zero-shot weight

NUM_CLASSES   = 234
HIDDEN_DIM    = 256
SR            = 32_000
CLIP_SAMPLES  = SR * 5   # 5-second clips

# Soundscapes: use test if available, else train
TEST_DIR = os.path.join(DATA_DIR, 'test_soundscapes')
SC_DIR   = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else os.path.join(DATA_DIR, 'train_soundscapes')
ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
print(f'Soundscapes: {len(ogg_files)} files in {SC_DIR}')

## Species Mapping (Perch → BirdCLEF 234 species)

In [ ]:
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = (bc_labels.reset_index()
             .rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1)
             .set_index('scientific_name'))
N_PERCH = len(bc_labels)

taxonomy       = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(N_PERCH).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

# Index for gathering from padded Perch label output; OOV → N_PERCH → 0 after pad
perch_indices = [int(mapping.loc[pl].iloc[0]) if pl in mapping.index else N_PERCH
                 for pl in primary_labels]
n_covered = sum(1 for i in perch_indices if i < N_PERCH)
print(f'Perch covers {n_covered}/{NUM_CLASSES} target species')

## Load Perch + Label-Head

In [ ]:
print('Loading Perch (CPU)...')
perch_model = tf.saved_model.load(PERCH_DIR)
sig = perch_model.signatures['serving_default']

# Build label-head: 234 → 256 → 234
class LabelHead(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.fc1 = tf.keras.layers.Dense(HIDDEN_DIM)
        self.act = tf.keras.layers.Activation('relu')
        self.fc2 = tf.keras.layers.Dense(NUM_CLASSES)
    def call(self, x, training=False):
        return self.fc2(self.act(self.fc1(x)))

head = LabelHead()
_ = head(tf.zeros((1, NUM_CLASSES)), training=False)  # build

with h5py.File(WEIGHTS_PATH, 'r') as wf:
    head.fc1.kernel.assign(wf['fc1']['vars']['0'][:])
    head.fc1.bias.assign(  wf['fc1']['vars']['1'][:])
    head.fc2.kernel.assign(wf['fc2']['vars']['0'][:])
    head.fc2.bias.assign(  wf['fc2']['vars']['1'][:])
print(f'Weights loaded <- {WEIGHTS_PATH}')

# Pre-compute perch_indices as TF constant for speed
TF_INDICES = tf.constant(perch_indices, dtype=tf.int32)

## Inference with ThreadPoolExecutor

In [ ]:
def predict_file(ogg_path):
    """Process one .ogg file. Returns (row_ids, predictions).
    Always returns 12 row_ids (5-60s). Skips inference if past TERMINATE_TIME.
    """
    ss_id   = re.sub(r'\.ogg$', '', os.path.basename(ogg_path), flags=re.IGNORECASE)
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]   # 12 row_ids per file
    zeros   = np.zeros((12, NUM_CLASSES), dtype=np.float32)

    if time.time() > TERMINATE_TIME:
        return row_ids, zeros

    try:
        audio, _ = librosa.load(ogg_path, sr=SR, mono=True)
        audio = audio.astype(np.float32)

        # Pad to multiple of CLIP_SAMPLES
        remainder = len(audio) % CLIP_SAMPLES
        if remainder:
            audio = np.pad(audio, (0, CLIP_SAMPLES - remainder))
        clips = audio.reshape(-1, CLIP_SAMPLES)   # (N_clips, 160000)
        n_clips = min(len(clips), 12)
        clips = clips[:n_clips]

        # Perch → label features (raw logits)
        out      = sig(inputs=tf.constant(clips))
        label    = tf.pad(out['label'], [[0, 0], [0, 1]])     # (N, N_PERCH+1)
        features = tf.gather(label, TF_INDICES, axis=1)        # (N, 234)

        # Ensemble: alpha * perch_zero_shot + (1-alpha) * label_head
        blended = (0.4 * tf.sigmoid(features)
                   + 0.6 * tf.sigmoid(head(tf.stop_gradient(features), training=False)))

        preds = blended.numpy()   # (N, 234)
        # Pad to 12 rows if file shorter than 60s
        if n_clips < 12:
            preds = np.vstack([preds, np.zeros((12 - n_clips, NUM_CLASSES), dtype=np.float32)])
        return row_ids, preds

    except Exception as e:
        print(f'ERROR {os.path.basename(ogg_path)}: {e}')
        return row_ids, zeros

In [ ]:
all_row_ids, all_preds = [], []

with ThreadPoolExecutor(max_workers=4) as executor:
    for k, (row_ids, preds) in enumerate(executor.map(predict_file, ogg_files)):
        all_row_ids.extend(row_ids)
        all_preds.append(preds)
        if k % 100 == 0:
            elapsed = (time.time() - START) / 60
            print(f'[{k+1}/{len(ogg_files)}] {elapsed:.1f} min elapsed')

print(f'Done: {len(all_row_ids)} rows in {(time.time()-START)/60:.1f} min')

## Build Submission

In [ ]:
# ── Post-processing: quantile_mix(0.5) + slide_max(0.5,9) — BEST +0.0185 ─────
sc_groups = {}
for i, rid in enumerate(all_row_ids):
    sc_groups.setdefault(rid.rsplit("_",1)[0], []).append(i)
for s in sc_groups:
    sc_groups[s].sort(key=lambda i: int(all_row_ids[i].rsplit("_",1)[1]))

def quantile_mix(preds, alpha=0.5):
    """Blend raw scores with rank-normalised scores (38th place BirdCLEF 2025)."""
    out = preds.copy()
    for idxs in sc_groups.values():
        chunk = preds[idxs]; T, C = chunk.shape
        rank_norm = np.zeros_like(chunk)
        for c in range(C):
            ranks = np.argsort(np.argsort(chunk[:, c]))
            rank_norm[:, c] = ranks / max(T-1, 1)
        out[idxs] = alpha*chunk + (1-alpha)*rank_norm
    return np.clip(out, 0.0, 1.0)

def sliding_max_blend(preds, w=0.5, window=9):
    out = preds.copy()
    for idxs in sc_groups.values():
        chunk = preds[idxs]; T = len(idxs)
        for t in range(T):
            lo, hi = max(0, t-window//2), min(T, t+window//2+1)
            out[idxs[t]] = w*chunk[t] + (1-w)*chunk[lo:hi].max(axis=0)
    return out

predictions = np.concatenate(all_preds, axis=0)
predictions = quantile_mix(predictions, alpha=0.5)
predictions = sliding_max_blend(predictions)
print(f"PP done. range [{predictions.min():.4f}, {predictions.max():.4f}]")


In [ ]:
submission  = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission  = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows x {len(primary_labels)} species')
print(f'Total elapsed: {(time.time()-START)/60:.1f} min')
submission.head(5)
